## 1.计算流通市值

In [2]:
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

capital_dt = pd.read_parquet('data/capital.parquet')
capital_dt=capital_dt.sort_values(by=['stock_code', 'change_date'], ascending=[True, True])
capital_dt.rename(columns={'change_date': 'trade_date'}, inplace=True)
price_dt = pd.read_parquet('data/eod_prices.parquet')
price_dt=price_dt.sort_values(by=['stock_code', 'trade_date'], ascending=[True, True])
mv_factor_dt=pd.merge(price_dt, capital_dt, on=['stock_code', 'trade_date'], how='left')
mv_factor_dt.fillna(method='ffill', inplace=True)
mv_factor_dt['float_a_shares_mv']=mv_factor_dt['float_a_shares']*mv_factor_dt['close_price']
mv_factor_dt.tail()


,stock_code,trade_date,close_price,adjusted_close,adj_factor,volume,turnover,total_shares,float_shares,total_a_shares,float_a_shares,float_a_shares_mv
12065642,920992.BJ,20251124,17.93,18.5230,1.033073,14658.45,26244.446,9674.163836,4740.627222,9673.609463,4739.70589,84982.926613
12065643,920992.BJ,20251125,18.05,18.6470,1.033073,10126.13,18280.210,9674.163836,4740.627222,9673.609463,4739.70589,85551.691320
12065644,920992.BJ,20251126,17.80,18.3887,1.033073,10567.74,19017.103,9674.163836,4740.627222,9673.609463,4739.70589,84366.764847
12065645,920992.BJ,20251127,17.65,18.2337,1.033073,7207.73,12856.675,9674.163836,4740.627222,9673.609463,4739.70589,83655.808964
12065646,920992.BJ,20251128,17.58,18.1614,1.033073,7501.27,13195.610,9674.163836,4740.627222,9673.609463,4739.70589,83324.029551


## 2.插入行业分类信息,计算市值因子
*流通市值 = close_price * float_a_shares*

(小市值策略应该看A股流通市值，直接跟盘子相关)

*市值因子 = -Ln(流通市值) 对于行业做标准化*

In [3]:
cat_df=pd.read_parquet('data/ref_data/Stock_Industry_Year.parquet')
mv_factor_dt['industry_name'] = mv_factor_dt.assign(year=pd.to_datetime(mv_factor_dt['trade_date'].astype(str)).dt.year).merge(cat_df[['stock_code', 'year', 'industry_name']], on=['stock_code', 'year'], how='left')['industry_name']
import numpy as np
mv_factor_dt.dropna(inplace=True)
mv_factor_dt['mv']=-np.log(mv_factor_dt['float_a_shares_mv'])
mv_factor_dt.head()

,stock_code,trade_date,close_price,adjusted_close,adj_factor,volume,turnover,total_shares,float_shares,total_a_shares,float_a_shares,float_a_shares_mv,industry_name,mv
102,000001.SZ,20130614,19.06,423.2233,22.204789,328553.36,6.251254e+05,512437.850182,310502.606910,512363.631057,310533.405247,5.918767e+06,银行,-15.593639
103,000001.SZ,20130617,19.24,427.2201,22.204789,309210.22,5.961902e+05,512437.850182,310502.606910,512363.631057,310533.405247,5.974663e+06,银行,-15.603038
104,000001.SZ,20130618,19.73,438.1005,22.204789,389599.38,7.701559e+05,512437.850182,310502.606910,512363.631057,310533.405247,6.126824e+06,银行,-15.628187
105,000001.SZ,20130619,19.24,427.2201,22.204789,369977.53,7.140700e+05,512437.850182,310502.606910,512363.631057,310533.405247,5.974663e+06,银行,-15.603038
106,000001.SZ,20130620,11.18,400.6981,35.840616,903066.54,1.029738e+06,819808.237690,496861.283537,819958.402672,496892.496412,5.555258e+06,银行,-15.530255


In [4]:
# 按「交易日期」分组，对当天全行业的mv列做Z-score标准化（(值-均值)/标准差）
mv_factor_dt['mv_standardized'] = mv_factor_dt.groupby('trade_date')['mv'].transform(
    lambda x: (x - x.mean()) / x.std() if x.std() != 0 else 0  # 避免标准差为0时除以0报错
)
mv_factor_dt.head()


,stock_code,trade_date,close_price,adjusted_close,adj_factor,volume,turnover,total_shares,float_shares,total_a_shares,float_a_shares,float_a_shares_mv,industry_name,mv,mv_standardized
102,000001.SZ,20130614,19.06,423.2233,22.204789,328553.36,6.251254e+05,512437.850182,310502.606910,512363.631057,310533.405247,5.918767e+06,银行,-15.593639,-1.744443
103,000001.SZ,20130617,19.24,427.2201,22.204789,309210.22,5.961902e+05,512437.850182,310502.606910,512363.631057,310533.405247,5.974663e+06,银行,-15.603038,-1.750973
104,000001.SZ,20130618,19.73,438.1005,22.204789,389599.38,7.701559e+05,512437.850182,310502.606910,512363.631057,310533.405247,6.126824e+06,银行,-15.628187,-1.774256
105,000001.SZ,20130619,19.24,427.2201,22.204789,369977.53,7.140700e+05,512437.850182,310502.606910,512363.631057,310533.405247,5.974663e+06,银行,-15.603038,-1.757097
106,000001.SZ,20130620,11.18,400.6981,35.840616,903066.54,1.029738e+06,819808.237690,496861.283537,819958.402672,496892.496412,5.555258e+06,银行,-15.530255,-1.726456


## 3.根据市值因子进行分位数多空选股

In [6]:
from scipy.stats.mstats import winsorize

# 读取数据）
fund_df = pd.read_parquet('data/ref_data/ETF_hold_512100.SH.parquet')

# ===================== 1. 数据预处理 =====================
# 处理mv_factor_dt的日期格式（数字转日期）
mv_factor_dt['trade_date'] = pd.to_datetime(mv_factor_dt['trade_date'].astype(str))

# 处理基金持仓数据的日期格式
fund_df = fund_df.rename(columns={'end_date': 'report_end_date'})
fund_df['report_end_date'] = pd.to_datetime(fund_df['report_end_date'])

# 生成基金持仓的有效时间区间（半年报：1-6月，年报：7-12月）
def get_position_valid_period(row):
    year = row['report_year']
    if row['report_type'] == '中报':
        start_date = pd.to_datetime(f'{year}-01-01')
        end_date = pd.to_datetime(f'{year}-06-30')
    elif row['report_type'] == '年报':
        start_date = pd.to_datetime(f'{year}-07-01')
        end_date = pd.to_datetime(f'{year}-12-31')
    else:
        start_date = end_date = row['report_end_date']
    return pd.Series([start_date, end_date], index=['pos_start_date', 'pos_end_date'])

fund_df[['pos_start_date', 'pos_end_date']] = fund_df.apply(get_position_valid_period, axis=1)

# 提取中证1000基金持仓的股票代码+有效区间（去重）
fund_holdings = fund_df[['stock_code', 'pos_start_date', 'pos_end_date']].drop_duplicates()

# ===================== 2. 核心选股逻辑（分位数多空） =====================
def select_stocks_by_date(date_group):
    trade_date = date_group.name
    daily_data = date_group.copy()
    
    # 筛选当日属于中证1000基金持仓的股票
    valid_holdings = fund_holdings[
        (fund_holdings['pos_start_date'] <= trade_date) &
        (fund_holdings['pos_end_date'] >= trade_date)
    ]['stock_code'].tolist()
    holding_stocks = daily_data[daily_data['stock_code'].isin(valid_holdings)]
    
    if len(holding_stocks) == 0:
        return pd.DataFrame()
    
    # 对mv列做1%/99%缩尾
    holding_stocks['mv_winsorized'] = winsorize(
        holding_stocks['mv_standardized'].values, 
        limits=(0.01, 0.01),
        inclusive=(True, True)
    )
    
    # 计算分位数并标记多空
    q10 = holding_stocks['mv_winsorized'].quantile(0.10)
    q90 = holding_stocks['mv_winsorized'].quantile(0.90)
    
    holding_stocks['signal'] = 'none'
    holding_stocks.loc[holding_stocks['mv_winsorized'] <= q10, 'signal'] = 'long'
    holding_stocks.loc[holding_stocks['mv_winsorized'] >= q90, 'signal'] = 'short'
    
    return holding_stocks

# 按交易日分组执行选股
selected_stocks = mv_factor_dt.groupby('trade_date').apply(select_stocks_by_date)

# 重置索引
selected_stocks = selected_stocks.reset_index(drop=True)

# 保留核心列（可根据需要调整）
final_selection_df = selected_stocks[['trade_date', 'stock_code', 'industry_name', 
                                       'mv', 'mv_winsorized', 'signal']]
final_selection_df.to_parquet('records/mv_selection_v3.parquet', index=False)


## 小市值单因子回测结果

In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from util import *

# ----------------------------
# 用户参数
# ----------------------------
INITIAL_CASH = 1_000_000          # 初始资金
START_DATE = '2017-01-01'         # 回测起始日
END_DATE   = '2025-06-30'         # 回测结束日
REBALANCE_FREQ = 5               # 每 N 个选股日调仓一次

PRICE_FILE = 'data/eod_prices.parquet'               # 价格数据路径
SELECTION_FILE = 'records/mv_selection_v3.parquet'   # 选股信号数据路径
FACTOR_FILE = 'records/factor_values.parquet'        # 因子值数据路径（用于 IC 计算）

# ----------------------------
# 加载数据
# ----------------------------
print("Loading data...")
price_df = load_and_preprocess_price(PRICE_FILE)                     # 自定义函数，返回价格df
selection_df = pd.read_parquet(SELECTION_FILE)                       # 读取选股信号

# 价格透视表：行=日期，列=股票代码，值=复权收盘价
price_pivot = price_df.pivot(index='trade_date', columns='stock_code', values='adjusted_close')
price_pivot = price_pivot.sort_index()
price_pivot.index = pd.to_datetime(price_pivot.index)                # 确保日期类型

# 筛选日期范围
price_pivot = price_pivot.loc[START_DATE:END_DATE]
selection_df = selection_df[(selection_df['trade_date'] >= START_DATE) &
                            (selection_df['trade_date'] <= END_DATE)]

# 获取所有有效的选股日期（有信号的交易日）
selection_dates = sorted(selection_df['trade_date'].unique())

# 确定调仓日期（每 REBALANCE_FREQ 个选股日取一个）
rebalance_dates = selection_dates[::REBALANCE_FREQ]
print(f"Number of rebalance dates: {len(rebalance_dates)}")

# 生成完整时间线（从第一个调仓日到结束日的所有交易日）
all_dates = price_pivot.index.tolist()
first_rebalance = rebalance_dates[0]
timeline = [d for d in all_dates if d >= first_rebalance]

# 加载因子数据（用于 IC 计算）
try:
    factor_df = pd.read_parquet(FACTOR_FILE)
    # 假设因子数据格式：trade_date, stock_code, factor_name, value
    # 若有多个因子，此处将展开为宽表或分组处理
    # 为简化，假设只有一个因子名为 'signal_factor'，或我们取第一个因子
    # 实际使用中请根据您的数据调整
    factor_pivot = factor_df.pivot(index='trade_date', columns='stock_code', values='value')
    factor_pivot = factor_pivot.sort_index()
    factor_pivot.index = pd.to_datetime(factor_pivot.index)
    # 仅保留回测时间范围内的因子
    factor_pivot = factor_pivot.loc[START_DATE:END_DATE]
    factor_available = True
    print("Factor data loaded successfully.")
except:
    print("Warning: Factor data not found. IC analysis will be skipped.")
    factor_available = False

# ----------------------------
# 辅助函数：获取调仓日信号
# ----------------------------
def get_signals(date):
    """返回给定日期对应的多头和空头股票列表"""
    day_data = selection_df[selection_df['trade_date'] == date]
    long_stocks = day_data[day_data['signal'] == 'long']['stock_code'].tolist()
    short_stocks = day_data[day_data['signal'] == 'short']['stock_code'].tolist()
    return long_stocks, short_stocks

# ========================
# 第一套回测：多空策略
# ========================
print("\n=== Running Long-Short Strategy ===")
cash = INITIAL_CASH
long_holdings = {}      # {股票代码: 股数}
short_holdings = {}     # {股票代码: 股数}
portfolio_values_ls = []

for date in timeline:
    today_prices = price_pivot.loc[date]

    if date in rebalance_dates:
        # 平仓
        for stock, shares in long_holdings.items():
            price = today_prices.get(stock, np.nan)
            if pd.notna(price):
                cash += shares * price
        for stock, shares in short_holdings.items():
            price = today_prices.get(stock, np.nan)
            if pd.notna(price):
                cash -= shares * price
        long_holdings = {}
        short_holdings = {}

        # 获取信号
        long_stocks, short_stocks = get_signals(date)

        # 过滤有效价格
        valid_long = [s for s in long_stocks if pd.notna(today_prices.get(s, np.nan))]
        valid_short = [s for s in short_stocks if pd.notna(today_prices.get(s, np.nan))]

        if valid_long or valid_short:
            # 资金分配：多空各占一半现金
            long_cash = cash / 2
            short_cash = cash / 2

            if valid_long:
                amount_per_long = long_cash / len(valid_long)
                for stock in valid_long:
                    price = today_prices[stock]
                    shares = amount_per_long / price
                    long_holdings[stock] = shares
                cash -= long_cash

            if valid_short:
                amount_per_short = short_cash / len(valid_short)
                for stock in valid_short:
                    price = today_prices[stock]
                    shares = amount_per_short / price
                    short_holdings[stock] = shares
                cash += short_cash   # 卖空获得现金

    # 每日估值
    long_mv = sum(shares * today_prices.get(stock, 0) for stock, shares in long_holdings.items())
    short_mv = sum(shares * today_prices.get(stock, 0) for stock, shares in short_holdings.items())
    portfolio_value = cash + long_mv - short_mv
    portfolio_values_ls.append(portfolio_value)

# 多空策略结果
portfolio_ls = pd.Series(portfolio_values_ls, index=timeline, name='Long-Short')
benchmark_series = compute_benchmark_returns(price_pivot, START_DATE, END_DATE)[1].reindex(timeline, method='ffill').fillna(1.0)
benchmark_funds_ls = INITIAL_CASH * benchmark_series
returns_ls = portfolio_ls.pct_change().dropna()
metrics_ls = compute_metrics(returns_ls, rf=0, periods_per_year=252)
print("\n--- Long-Short Metrics ---")
for k, v in metrics_ls.items():
    print(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")

# ========================
# 第二套回测：仅做多策略
# ========================
print("\n=== Running Long-Only Strategy ===")
cash = INITIAL_CASH
long_holdings_only = {}
portfolio_values_lo = []

for date in timeline:
    today_prices = price_pivot.loc[date]

    if date in rebalance_dates:
        # 平仓多头
        for stock, shares in long_holdings_only.items():
            price = today_prices.get(stock, np.nan)
            if pd.notna(price):
                cash += shares * price
        long_holdings_only = {}

        # 获取多头信号
        long_stocks, _ = get_signals(date)
        valid_long = [s for s in long_stocks if pd.notna(today_prices.get(s, np.nan))]

        if valid_long:
            # 全额投入多头
            amount_per_long = cash / len(valid_long)
            for stock in valid_long:
                price = today_prices[stock]
                shares = amount_per_long / price
                long_holdings_only[stock] = shares
            cash = 0   # 全部买入，现金归零

    # 每日估值
    long_mv = sum(shares * today_prices.get(stock, 0) for stock, shares in long_holdings_only.items())
    portfolio_value = cash + long_mv
    portfolio_values_lo.append(portfolio_value)

portfolio_lo = pd.Series(portfolio_values_lo, index=timeline, name='Long-Only')
returns_lo = portfolio_lo.pct_change().dropna()
metrics_lo = compute_metrics(returns_lo, rf=0, periods_per_year=252)
print("\n--- Long-Only Metrics ---")
for k, v in metrics_lo.items():
    print(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")

# ========================
# 仅做多策略的 IC 与换手率分析
# ========================
if factor_available:
    print("\n=== IC and Turnover Analysis for Long-Only Strategy ===")
    
    # 计算未来一个月收益率（假设每月最后一个交易日为调仓点，这里使用月度频率）
    # 我们首先需要将因子值与未来收益率对齐。为简化，我们以每月最后一个交易日作为 IC 计算日。
    # 获取所有月份的最后交易日
    month_end_dates = price_pivot.resample('M').last().index
    
    # 准备存储 IC 数据
    ic_series = []
    # 若存在多个因子，可循环处理；这里假设只有一个因子，且因子数据已存在 factor_pivot
    # 如果您的因子数据包含多个因子，请修改此处
    factor_name = 'signal_factor'  # 假设的因子名称
    # 假设 factor_pivot 的列是股票代码，行是日期，值为因子值
    
    for date in month_end_dates:
        # 确保日期在 factor_pivot 和 price_pivot 范围内
        if date not in factor_pivot.index or date not in price_pivot.index:
            continue
        
        # 获取该日因子值
        factor_today = factor_pivot.loc[date].dropna()
        if factor_today.empty:
            continue
        
        # 计算未来一个月收益率：从 date 的下一个交易日到 date+1M 的累计收益
        # 获取下一个月末的日期
        next_month_end = date + pd.offsets.MonthEnd(1)
        # 获取该期间所有交易日价格
        if next_month_end not in price_pivot.index:
            next_month_end = price_pivot.index[price_pivot.index >= next_month_end].min()
        # 计算期间收益率：未来价格 / 当前价格 - 1
        future_prices = price_pivot.loc[next_month_end]
        current_prices = price_pivot.loc[date]
        # 对齐股票
        common_stocks = factor_today.index.intersection(current_prices.dropna().index).intersection(future_prices.dropna().index)
        if len(common_stocks) < 5:
            continue
        factor_vals = factor_today.loc[common_stocks]
        rets = future_prices.loc[common_stocks] / current_prices.loc[common_stocks] - 1
        
        # 计算 IC（相关系数）
        ic = factor_vals.corr(rets)
        ic_series.append((date, ic))
    
    if ic_series:
        ic_df = pd.DataFrame(ic_series, columns=['date', 'ic']).set_index('date')
        ic_df['ic'].plot(title='Monthly IC Time Series', figsize=(10, 6))
        plt.xlabel('Date')
        plt.ylabel('IC')
        plt.grid(True)
        plt.show()
        
        # 计算 ICIR
        mean_ic = ic_df['ic'].mean()
        std_ic = ic_df['ic'].std()
        icir = mean_ic / std_ic if std_ic != 0 else np.nan
        print(f"Mean IC: {mean_ic:.4f}")
        print(f"Std IC:  {std_ic:.4f}")
        print(f"ICIR:    {icir:.4f}")
    else:
        print("No valid IC data computed.")
    
    # 计算因子 top-quantile 的月度换手率
    # 这里我们取因子值前 20% 作为 top-quantile
    turnover_series = []
    prev_top_stocks = set()
    for date in month_end_dates:
        if date not in factor_pivot.index:
            continue
        factor_today = factor_pivot.loc[date].dropna()
        if factor_today.empty:
            continue
        # 按因子值降序排列，取前 20%
        quantile = 0.2
        n_top = max(1, int(len(factor_today) * quantile))
        top_stocks = set(factor_today.nlargest(n_top).index)
        
        if prev_top_stocks:
            # 换手率 = 新集合与旧集合的对称差大小 / 旧集合大小
            # 常见定义：换手率 = (新集合并集 - 新集合交集) / 旧集合大小
            # 这里简单计算
            turnover = len(top_stocks.symmetric_difference(prev_top_stocks)) / len(prev_top_stocks)
            turnover_series.append((date, turnover))
        prev_top_stocks = top_stocks
    
    if turnover_series:
        turnover_df = pd.DataFrame(turnover_series, columns=['date', 'turnover']).set_index('date')
        print("\nMonthly Turnover for Top Quantile:")
        print(turnover_df.describe())
        turnover_df.plot(title='Monthly Turnover of Factor Top Quantile', figsize=(10, 6))
        plt.xlabel('Date')
        plt.ylabel('Turnover')
        plt.grid(True)
        plt.show()
    else:
        print("No turnover data computed.")
else:
    print("Skipping IC and turnover analysis due to missing factor data.")

# ========================
# 绘制两策略对比图
# ========================
plot_results(portfolio_ls, benchmark_funds_ls, timeline, title='Long-Short Strategy')
plot_results(portfolio_lo, benchmark_funds_ls, timeline, title='Long-Only Strategy')

Loading data...
Number of rebalance dates: 412

=== Running Long-Short Strategy ===

--- Long-Short Metrics ---
Annualized Return: -0.0401
Volatility: 0.0749
Sharpe Ratio: -0.5347
Max Drawdown: -0.3749

=== Running Long-Only Strategy ===

--- Long-Only Metrics ---
Annualized Return: -0.0621
Volatility: 0.2433
Sharpe Ratio: -0.2555
Max Drawdown: -0.5727
Skipping IC and turnover analysis due to missing factor data.


In [ ]:
# 股息因子

In [12]:
from warnings import filterwarnings
filterwarnings('ignore')
import pandas as pd
dividend = pd.read_parquet('data/dividend.parquet')
dividend = dividend.sort_values(by=['stock_code', 'ex_date'], ascending=[True, True])
dividend.head(10)

,stock_code,announce_date,ex_date,div_pre_tax,div_after_tax,progress,base_shares,div_year,currency,is_no_div,total_cash_div
8165,000001.SZ,20130614,20130620,0.170001,0.131472,3,5.123809e+05,20121231,CNY,0,8.709306e+08
6411,000001.SZ,20140606,20140612,0.159974,0.151981,3,9.521493e+05,20131231,CNY,0,1.523397e+09
20151,000001.SZ,20150407,20150413,0.173981,0.165298,3,1.142486e+06,20141231,CNY,0,1.988210e+09
21055,000001.SZ,20160608,20160616,0.152994,0.152997,3,1.430813e+06,20151231,CNY,0,2.189083e+09
3884,000001.SZ,20170717,20170721,0.157993,0.158005,3,1.717085e+06,20161231,CNY,0,2.712675e+09
6258,000001.SZ,20180706,20180712,0.135999,0.136014,3,1.717202e+06,20171231,CNY,0,2.335101e+09
16687,000001.SZ,20190620,20190626,0.144977,0.145004,3,1.717122e+06,20181231,CNY,0,2.489544e+09
13074,000001.SZ,20200522,20200528,0.218038,0.218027,3,1.940372e+06,20191231,CNY,0,4.230812e+09
26102,000001.SZ,20210507,20210514,0.179976,0.179985,3,1.940534e+06,20201231,CNY,0,3.493105e+09
21192,000001.SZ,20220715,20220722,0.227990,0.227989,3,1.940405e+06,20211231,CNY,0,4.425158e+09


## 1.计算股息因子

预期股息率（TTM）：最近365个自然日div_pre_tax和 / **不复权**市场价格，计入日期区间所有announce_date在这之间的总分红；
静态股息率：上一财年总div_pre_tax和 / **不复权**市场价格，计入ex_date在当前日期之前且div_year为当前日期上一年的总分红；
动态股息率（TTM）：最近365个自然日div_pre_tax和 / **不复权**市场价格，计入日期区间所有ex_date在这之间的总分红；
~~div_after_tax~~ div_pre_tax ：税前股息反映的是公司真实分红能力，税后股息只反映单个投资者的到手收益，前者更适合做因子/估值。

In [13]:
price_dt = pd.read_parquet('data/eod_prices.parquet')
price_dt=price_dt.sort_values(by=['stock_code', 'trade_date'], ascending=[True, True])
price_dt.head()


,stock_code,trade_date,close_price,adjusted_close,adj_factor,volume,turnover
2983,000001.SZ,20130104,15.99,355.0546,22.204789,443851.37,717567.5466
8347,000001.SZ,20130107,16.30,361.9381,22.204789,357169.25,578450.4876
10530,000001.SZ,20130108,16.00,355.2766,22.204789,312479.12,501360.0937
13553,000001.SZ,20130109,15.86,352.1680,22.204789,251329.15,399696.1831
10100,000001.SZ,20130110,15.87,352.3900,22.204789,240030.27,383347.7326


In [ ]:
import pandas as pd
from util import calc_factors
from tqdm import tqdm

dividend['announce_date'] = pd.to_datetime(dividend['announce_date'])
dividend['ex_date'] = pd.to_datetime(dividend['ex_date'])
price_dt['trade_date'] = pd.to_datetime(price_dt['trade_date'])

price_dt = price_dt.sort_values(['stock_code', 'trade_date'])
dividend = dividend.sort_values(['stock_code', 'announce_date'])

df_factors_list = []

# 获取总股票数
total_stocks = price_dt['stock_code'].nunique()

# 使用 tqdm 包装 groupby 迭代
for stock, price_sub in tqdm(price_dt.groupby('stock_code'), total=total_stocks, desc="Processing stocks"):
    if stock in dividend['stock_code'].values:
        div_sub = dividend[dividend['stock_code'] == stock]
        factor_sub = calc_factors(price_sub, div_sub)
        df_factors_list.append(factor_sub)
    else:
        price_sub = price_sub.copy()
        price_sub['expected_div_yield'] = 0.0
        price_sub['static_div_yield'] = 0.0
        price_sub['dynamic_div_yield'] = 0.0
        df_factors_list.append(price_sub[['stock_code', 'trade_date', 
                                          'expected_div_yield', 'static_div_yield', 'dynamic_div_yield']])

df_factors = pd.concat(df_factors_list, ignore_index=True)
df_factors.to_parquet('records/div_temp.parquet', index=False)


Processing stocks:   2%|▏         | 97/5693 [05:25<5:19:45,  3.43s/it]

In [ ]:
df_factors.tail(10)

,stock_code,trade_date,expected_div_yield,static_div_yield,dynamic_div_yield
12065627,920992.BJ,2025-11-03,0.003737,0.0,0.003737
12065628,920992.BJ,2025-11-04,0.003795,0.0,0.003795
12065629,920992.BJ,2025-11-05,0.003756,0.0,0.003756
12065630,920992.BJ,2025-11-06,0.003830,0.0,0.003830
12065631,920992.BJ,2025-11-07,0.003901,0.0,0.003901
12065632,920992.BJ,2025-11-10,0.003885,0.0,0.003885
12065633,920992.BJ,2025-11-11,0.003937,0.0,0.003937
12065634,920992.BJ,2025-11-12,0.003920,0.0,0.003920
12065635,920992.BJ,2025-11-13,0.003918,0.0,0.003918
12065636,920992.BJ,2025-11-14,0.003947,0.0,0.003947


## 2.分位数选股

In [ ]:
import pandas as pd
from scipy.stats.mstats import winsorize
import os

# ===================== 1. 数据预处理 =====================
# 假设 df_factors 已存在，包含列：stock_code, trade_date, dynamic_div_yield
# 确保 trade_date 为 datetime 类型（如果不是则转换）
df_factors['trade_date'] = pd.to_datetime(df_factors['trade_date'])

# 可选：去除股息率为 NaN 或 0 的股票（根据业务逻辑决定）
# df_factors = df_factors[df_factors['dynamic_div_yield'] > 0]

# ===================== 2. 核心选股逻辑 =====================
def select_stocks_by_date(date_group):
    """
    对单个日期的样本执行选股：
    1. （可选）对 dynamic_div_yield 做 1%/99% 缩尾
    2. 按缩尾后股息率排序，取前10%股票（做多）和后10%股票（做空）
    3. 返回带有信号标记的 DataFrame
    """
    trade_date = date_group.name
    daily_data = date_group.copy()
    
    # 剔除股息率为空或无效的股票
    daily_data = daily_data.dropna(subset=['dynamic_div_yield'])
    
    if len(daily_data) == 0:
        return pd.DataFrame()
    
    # 缩尾处理（1%和99%分位数）
    daily_data['div_yield_winsorized'] = winsorize(
        daily_data['dynamic_div_yield'].values,
        limits=(0.01, 0.01),
        inclusive=(True, True)
    )
    
    # 按缩尾后股息率降序排序
    daily_data_sorted = daily_data.sort_values('div_yield_winsorized', ascending=False)
    n = len(daily_data_sorted)
    n_long = max(1, int(n * 0.1))   # 前10%做多（至少1只）
    n_short = max(1, int(n * 0.1))  # 后10%做空（至少1只）
    
    # 选取前n_long只股票（做多）
    long_stocks = daily_data_sorted.head(n_long).copy()
    long_stocks['signal'] = 'long'
    
    # 选取后n_short只股票（做空）
    short_stocks = daily_data_sorted.tail(n_short).copy()
    short_stocks['signal'] = 'short'
    
    # 合并结果
    selected = pd.concat([long_stocks, short_stocks], ignore_index=True)
    
    # 可选：添加排名（如果需要，可保留）
    # selected['rank'] = range(1, len(selected) + 1)
    
    return selected

# 按交易日分组执行选股
selected_stocks = df_factors.groupby('trade_date', group_keys=False).apply(select_stocks_by_date)

# 重置索引（清理groupby后的索引）
selected_stocks = selected_stocks.reset_index(drop=True)

# 保留核心列（可自行调整）
final_selection_df = selected_stocks[['trade_date', 'stock_code', 'dynamic_div_yield', 'div_yield_winsorized', 'signal']]

# ===================== 3. 保存结果 =====================
# 确保保存目录存在
os.makedirs('records', exist_ok=True)

# 保存为 Parquet 格式（推荐使用 pyarrow 引擎）
final_selection_df.to_parquet('records/div_selection_v2.parquet', index=False)

print(f"选股完成，共 {len(final_selection_df)} 条记录，保存至 records/div_selection_v2.parquet")

## 3.回测结果

In [ ]:
import pandas as pd
import numpy as np
from util import *   # 假设 util 包含 load_and_preprocess_price, compute_benchmark_returns, compute_metrics, plot_results

# ----------------------------
# 用户参数
# ----------------------------
INITIAL_CASH = 1_000_000          # 初始资金
START_DATE = '2017-01-01'         # 回测起始日
END_DATE   = '2025-06-30'         # 回测结束日
REBALANCE_FREQ = 20               # 每 N 个选股日调仓一次

PRICE_FILE = 'data/eod_prices.parquet'               # 价格数据路径
SELECTION_FILE = 'records/div_selection_v2.parquet'  # 多空选股数据路径

# ----------------------------
# 加载数据
# ----------------------------
print("Loading data...")
price_df = load_and_preprocess_price(PRICE_FILE)                     # 自定义函数，返回价格df
selection_df = pd.read_parquet(SELECTION_FILE)                       # 读取选股信号

# 价格透视表：行=日期，列=股票代码，值=复权收盘价
price_pivot = price_df.pivot(index='trade_date', columns='stock_code', values='adjusted_close')
price_pivot = price_pivot.sort_index()
price_pivot.index = pd.to_datetime(price_pivot.index)                # 确保日期类型

# 筛选日期范围
price_pivot = price_pivot.loc[START_DATE:END_DATE]
selection_df = selection_df[(selection_df['trade_date'] >= START_DATE) &
                            (selection_df['trade_date'] <= END_DATE)]

# 获取所有有效的选股日期（有信号的交易日）
selection_dates = sorted(selection_df['trade_date'].unique())

# 确定调仓日期（每 REBALANCE_FREQ 个选股日取一个）
rebalance_dates = selection_dates[::REBALANCE_FREQ]
print(f"Number of rebalance dates: {len(rebalance_dates)}")

# 生成完整时间线（从第一个调仓日到结束日的所有交易日）
all_dates = price_pivot.index.tolist()
first_rebalance = rebalance_dates[0]
timeline = [d for d in all_dates if d >= first_rebalance]

# ----------------------------
# 辅助函数：获取调仓日的多空信号
# ----------------------------
def get_signals(date):
    """返回给定日期对应的多头和空头股票列表"""
    day_data = selection_df[selection_df['trade_date'] == date]
    long_stocks = day_data[day_data['signal'] == 'long']['stock_code'].tolist()
    short_stocks = day_data[day_data['signal'] == 'short']['stock_code'].tolist()
    return long_stocks, short_stocks

# ----------------------------
# 计算基准收益（全市场等权重）
# ----------------------------
benchmark_daily, benchmark_cum = compute_benchmark_returns(price_pivot, START_DATE, END_DATE)
benchmark_series = benchmark_cum.reindex(timeline, method='ffill').fillna(1.0)
benchmark_funds = INITIAL_CASH * benchmark_series

# ========================
# 1. 多空策略
# ========================
print("\n=== Running Long-Short Strategy ===")
cash_ls = INITIAL_CASH
long_holdings = {}      # {stock: shares}
short_holdings = {}     # {stock: shares}
portfolio_values_ls = []
turnover_data_ls = []   # 记录每月换手率

for date in timeline:
    today_prices = price_pivot.loc[date]

    # --- 调仓日 ---
    if date in rebalance_dates:
        # 平仓多头
        for stock, shares in long_holdings.items():
            price = today_prices.get(stock, np.nan)
            if pd.notna(price):
                cash_ls += shares * price
        # 平仓空头（买入平仓）
        for stock, shares in short_holdings.items():
            price = today_prices.get(stock, np.nan)
            if pd.notna(price):
                cash_ls -= shares * price
        long_holdings.clear()
        short_holdings.clear()

        # 获取新信号
        long_stocks, short_stocks = get_signals(date)

        # 过滤有效价格股票
        valid_long = [s for s in long_stocks if pd.notna(today_prices.get(s, np.nan))]
        valid_short = [s for s in short_stocks if pd.notna(today_prices.get(s, np.nan))]

        if valid_long or valid_short:
            # 资金分配：多头和空头各占当前现金的一半（市场中性）
            long_cash = cash_ls / 2
            short_cash = cash_ls / 2

            # 构建多头（等权重）
            if valid_long:
                amount_per_long = long_cash / len(valid_long)
                for stock in valid_long:
                    price = today_prices[stock]
                    shares = amount_per_long / price
                    long_holdings[stock] = shares
                cash_ls -= long_cash

            # 构建空头（等权重）
            if valid_short:
                amount_per_short = short_cash / len(valid_short)
                for stock in valid_short:
                    price = today_prices[stock]
                    shares = amount_per_short / price
                    short_holdings[stock] = shares
                cash_ls += short_cash  # 卖空获得现金

            # 计算该次调仓的换手率（交易金额 / 调仓前总资产）
            # 调仓前总资产 = cash_ls（已平仓后的现金） + 原持仓市值（已清仓，所以为0）
            # 简化：换手率 = (多头买入金额 + 空头卖出市值) / 调仓前资产
            pre_rebalance_asset = cash_ls - short_cash  # 调仓前现金（不含新空头获得的现金）
            # 实际交易金额：多头买入 long_cash + 空头卖出市值 short_cash
            turnover = (long_cash + short_cash) / pre_rebalance_asset if pre_rebalance_asset > 0 else 0
            turnover_data_ls.append((date, turnover))

    # --- 每日估值 ---
    long_mv = sum(shares * today_prices.get(stock, 0) for stock, shares in long_holdings.items())
    short_mv = sum(shares * today_prices.get(stock, 0) for stock, shares in short_holdings.items())
    portfolio_value = cash_ls + long_mv - short_mv
    portfolio_values_ls.append(portfolio_value)

# 多空策略结果
portfolio_ls = pd.Series(portfolio_values_ls, index=timeline, name='Long-Short')
returns_ls = portfolio_ls.pct_change().dropna()
metrics_ls = compute_metrics(returns_ls, rf=0, periods_per_year=252)
print("\n--- Long-Short Metrics ---")
for k, v in metrics_ls.items():
    print(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")

# 计算月度换手率（多空）
if turnover_data_ls:
    turnover_df_ls = pd.DataFrame(turnover_data_ls, columns=['date', 'turnover']).set_index('date')
    # 转换为月度频率：取每月最后一天的换手率（若该月无调仓，则换手率为0）
    monthly_turnover_ls = turnover_df_ls.resample('M').last().fillna(0)
    print("\n--- Long-Strategy Monthly Turnover ---")
    print(monthly_turnover_ls.describe())
    # 绘制换手率时序图（可选）
    # monthly_turnover_ls.plot(title='Long-Short Strategy Monthly Turnover')
else:
    print("No turnover data for long-short strategy.")

# ========================
# 2. 仅做多策略
# ========================
print("\n=== Running Long-Only Strategy ===")
cash_lo = INITIAL_CASH
holdings_lo = {}        # {stock: shares}
portfolio_values_lo = []
turnover_data_lo = []

for date in timeline:
    today_prices = price_pivot.loc[date]

    if date in rebalance_dates:
        # 平仓
        for stock, shares in holdings_lo.items():
            price = today_prices.get(stock, np.nan)
            if pd.notna(price):
                cash_lo += shares * price
        holdings_lo.clear()

        # 获取多头信号
        long_stocks, _ = get_signals(date)
        valid_long = [s for s in long_stocks if pd.notna(today_prices.get(s, np.nan))]

        if valid_long:
            # 计算调仓前资产
            pre_asset = cash_lo
            # 买入
            amount_per_stock = cash_lo / len(valid_long)
            for stock in valid_long:
                price = today_prices[stock]
                shares = amount_per_stock / price
                holdings_lo[stock] = shares
            cash_lo = 0.0
            # 换手率 = 买入金额 / 调仓前资产（因为全部现金投入）
            turnover = pre_asset / pre_asset if pre_asset > 0 else 0
            turnover_data_lo.append((date, turnover))

    # 每日估值
    long_mv = sum(shares * today_prices.get(stock, 0) for stock, shares in holdings_lo.items())
    portfolio_value = cash_lo + long_mv
    portfolio_values_lo.append(portfolio_value)

portfolio_lo = pd.Series(portfolio_values_lo, index=timeline, name='Long-Only')
returns_lo = portfolio_lo.pct_change().dropna()
metrics_lo = compute_metrics(returns_lo, rf=0, periods_per_year=252)
print("\n--- Long-Only Metrics ---")
for k, v in metrics_lo.items():
    print(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")

# 计算月度换手率（仅做多）
if turnover_data_lo:
    turnover_df_lo = pd.DataFrame(turnover_data_lo, columns=['date', 'turnover']).set_index('date')
    monthly_turnover_lo = turnover_df_lo.resample('M').last().fillna(0)
    print("\n--- Long-Only Strategy Monthly Turnover ---")
    print(monthly_turnover_lo.describe())
else:
    print("No turnover data for long-only strategy.")

# ----------------------------
# 绘制净值曲线对比
# ----------------------------
import matplotlib.pyplot as plt
plt.figure(figsize=(12, 6))
plt.plot(portfolio_ls.index, portfolio_ls.values, label='Long-Short', linewidth=1.5)
plt.plot(portfolio_lo.index, portfolio_lo.values, label='Long-Only', linewidth=1.5)
plt.plot(benchmark_funds.index, benchmark_funds.values, label='Benchmark', linewidth=1.5, linestyle='--')
plt.title('Strategy Comparison')
plt.xlabel('Date')
plt.ylabel('Portfolio Value')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# 也可使用原 plot_results 分别绘制（已包含在 util 中，此处可选）
# plot_results(portfolio_ls, benchmark_funds, timeline, title='Long-Short Strategy')
# plot_results(portfolio_lo, benchmark_funds, timeline, title='Long-Only Strategy')